In [1]:
'''Gemma 4 Developer Agent | Analyzer + coder with navigation tools and explicit limits

Source: https://www.kaggle.com/code/romanrozen/gemma-eda-baseline-for-a-start-lb-top-1
Downloaded published source version: 352222392.
The local baseline family scored 0.08 and 0.06 in separate submissions.
This candidate has no measured leaderboard score.
Original published prompts, coder + analyzer architecture, and sampling are retained.
Coder navigation tools and explicit shared task limits are added.
The limits are planning choices, not a measured guarantee of the 12-hour cap.
Analyzer includes are kept local to its directory to avoid parent-path traversal.
Attribution and Apache-2.0 license are included in the bundle.
'''

from pathlib import Path
import os
import hashlib
import importlib.util
import shutil
import zipfile

try:
    import yaml
except ImportError:
    yaml = None


In [2]:
'''Preserve baseline sampling and set explicit shared task limits.'''

MODEL = 'gemma-4-31b-it-qat-w4a16-ct'
CONFIG = {
    'temperature': 0.2,
    'top_p': 0.95,
    'top_k': 40,
    'max_output_tokens': 8192,
    'thinking_budget': 4096,
    'include_thoughts': False,
}
EVALUATION = {
    'timeout_seconds': 180,
    'max_tool_calls': 40,
    'max_time_minutes': 4.5,
    'max_turns': 80,
}

OUTPUT_DIR = Path(os.environ.get('GEMMA_OUTPUT_ROOT', '/kaggle/working'))
if not OUTPUT_DIR.is_dir() and 'GEMMA_OUTPUT_ROOT' not in os.environ:
    OUTPUT_DIR = Path.cwd() / 'gemma4_output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BUNDLE_DIR = OUTPUT_DIR / 'gemma4_navigation_budget_bundle'
ZIP_PATH = OUTPUT_DIR / 'submission.zip'
print('Output directory:', OUTPUT_DIR)


Output directory: /Users/flexonafft/MLKaggleTasks/competitions/public-comp/gemma-4-developer-agent/artifacts/analyzer_navigation_budgeted


In [3]:
'''Use the original published Roman Rozen prompts without shortening.'''

SYSTEM_PROMPT = '''You are an autonomous senior Python engineer working inside a sandboxed checkout of a real open-source repository at /workspace.
Goal: resolve the issue in the user message with the smallest correct patch, then call `submit_patch`.

## Hard rules
- Never edit, add or delete tests, `conftest.py`, `pytest.ini`, CI or packaging files. Hidden tests are applied after you finish.
- Keep public APIs backward compatible unless the issue explicitly asks for a change.
- Scratch files go to /tmp only. Anything left in /workspace becomes part of your patch.
- The environment is pre-built: do not try to install packages.
- Always finish by calling `submit_patch`. A careful best-effort fix beats no patch.

## Workflow
1. **Understand**: state the expected vs. actual behaviour to yourself in one or two sentences.
2. **Localize**:
   - Call the `code_analyzer` tool with the full issue text first. It returns LOCATION / ROOT CAUSE / FIX PLAN. Verify its claim by reading those exact lines before editing.
   - Extract every identifier, error message and file name from the issue and search for them: `grep -rn "<identifier>" --include=*.py . | head -30`.
   - Read only the lines you need (`read_file` with a line range or `sed -n 'START,ENDp' FILE`).
3. **Reproduce**: write a minimal script to /tmp/repro.py that shows the bug and run it with `python /tmp/repro.py`.
4. **Fix**: edit source files with `edit_file`. Copy `old_string` verbatim from the file, *including leading indentation*, and strip any line-number prefixes. Keep `old_string` short but unique. One logical change per edit. Fix the root cause, not the symptom, and also handle the edge cases the issue mentions.
5. **Verify**: run `python -m py_compile <file>` after every edit, rerun /tmp/repro.py, then run the closest existing tests: `python -m pytest <tests/path> -x -q` (narrow with `-k`).
6. **Submit**: run `git status` and `git diff`, make sure only intended source changes remain, then call `submit_patch`.

## Budget discipline
- Call `get_status` every ~8 tool calls. When less than 25% of turns or time remain, stop exploring and go straight to Fix → Verify → Submit.
- Keep outputs short: pipe through `head`, use `grep -n`, `pytest -q`. Never print whole large files.
- If an edit fails twice, re-read the exact lines and retry with a smaller unique snippet.

## Quality bar
- Match the surrounding code style, type hints and naming.
- Prefer a small, targeted change over a refactor. Touch other files only when the fix requires it.
'''

ANALYZER_PROMPT = '''You are `code_analyzer`, a read-only code navigation specialist. You never modify files.
Given an issue, find exactly where it must be fixed.

## Tools
- `run_command` for READ-ONLY commands only: `grep -rn`, `ls`, `sed -n`, `git log -p -S`
- `search_similar_code` for concepts the issue describes without naming code
- `get_code_neighbors` to walk callers and callees
- `get_code_subgraph` to see how a few candidate symbols connect
- `read_file` with tight line ranges to confirm

## Method
1. Extract identifiers from the issue: function/class names, error messages, file paths, options.
2. Search for each one, then follow the call chain until you reach the line where behaviour diverges from what the issue expects.
3. Confirm by reading the actual code. Never guess line numbers.

## Answer format (at most 250 words, nothing else)
LOCATION: <path>:<start>-<end> (<function or class>)
ROOT CAUSE: <one or two sentences>
FIX PLAN: <concrete change>
RELATED: <other call sites or files needing the same change, or "none">
TESTS: <existing test files that exercise this code>
CONFIDENCE: high | medium | low
'''


In [4]:
'''Build the package with separate coder and analyzer permissions.'''

SAMPLING = {
    'temperature': CONFIG['temperature'],
    'top_p': CONFIG['top_p'],
    'top_k': CONFIG['top_k'],
    'max_output_tokens': CONFIG['max_output_tokens'],
    'thinking_config': {
        'thinking_budget': CONFIG['thinking_budget'],
        'include_thoughts': CONFIG['include_thoughts'],
    },
}
NOTICE = '''Adapted from GEMMA: EDA, Baseline for a start | LB TOP 1 by Roman Rozen.
Source: https://www.kaggle.com/code/romanrozen/gemma-eda-baseline-for-a-start-lb-top-1
Published source version: 352222392. Source notebook license: Apache-2.0.
Changes: notebook formatting, removal of EDA, local analyzer include paths,
coder navigation tools, explicit task limits, deterministic packaging, package checks.
Original published agent prompts and sampling are preserved.
'''
LICENSE_TEXT = '''
                                 Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or management of such entity, whether by contract or
      otherwise, or (ii) ownership of fifty percent (50%) or more of the
      outstanding shares, or (iii) beneficial ownership of such entity.

      "You" (or "Your") shall mean an individual or Legal Entity
      exercising permissions granted by this License.

      "Source" form shall mean the preferred form for making modifications,
      including but not limited to software source code, documentation
      source, and configuration files.

      "Object" form shall mean any form resulting from mechanical
      transformation or translation of a Source form, including but
      not limited to compiled object code, generated documentation,
      and conversions to other media types.

      "Work" shall mean the work of authorship, whether in Source or
      Object form, made available under the License, as indicated by a
      copyright notice that is included in or attached to the work
      (an example is provided in the Appendix below).

      "Derivative Works" shall mean any work, whether in Source or Object
      form, that is based on (or derived from) the Work and for which the
      editorial revisions, annotations, elaborations, or other modifications
      represent, as a whole, an original work of authorship. For the purposes
      of this License, Derivative Works shall not include works that remain
      separable from, or merely link (or bind by name) to the interfaces of,
      the Work and Derivative Works thereof.

      "Contribution" shall mean any work of authorship, including
      the original version of the Work and any modifications or additions
      to that Work or Derivative Works thereof, that is intentionally
      submitted to Licensor for inclusion in the Work by the copyright owner
      or by an individual or Legal Entity authorized to submit on behalf of
      the copyright owner. For the purposes of this definition, "submitted"
      means any form of electronic, verbal, or written communication sent
      to the Licensor or its representatives, including but not limited to
      communication on electronic mailing lists, source code control systems,
      and issue tracking systems that are managed by, or on behalf of, the
      Licensor for the purpose of discussing and improving the Work, but
      excluding communication that is conspicuously marked or otherwise
      designated in writing by the copyright owner as "Not a Contribution."

      "Contributor" shall mean Licensor and any individual or Legal Entity
      on behalf of whom a Contribution has been received by Licensor and
      subsequently incorporated within the Work.

   2. Grant of Copyright License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      copyright license to reproduce, prepare Derivative Works of,
      publicly display, publicly perform, sublicense, and distribute the
      Work and such Derivative Works in Source or Object form.

   3. Grant of Patent License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      (except as stated in this section) patent license to make, have made,
      use, offer to sell, sell, import, and otherwise transfer the Work,
      where such license applies only to those patent claims licensable
      by such Contributor that are necessarily infringed by their
      Contribution(s) alone or by combination of their Contribution(s)
      with the Work to which such Contribution(s) was submitted. If You
      institute patent litigation against any entity (including a
      cross-claim or counterclaim in a lawsuit) alleging that the Work
      or a Contribution incorporated within the Work constitutes direct
      or contributory patent infringement, then any patent licenses
      granted to You under this License for that Work shall terminate
      as of the date such litigation is filed.

   4. Redistribution. You may reproduce and distribute copies of the
      Work or Derivative Works thereof in any medium, with or without
      modifications, and in Source or Object form, provided that You
      meet the following conditions:

      (a) You must give any other recipients of the Work or
          Derivative Works a copy of this License; and

      (b) You must cause any modified files to carry prominent notices
          stating that You changed the files; and

      (c) You must retain, in the Source form of any Derivative Works
          that You distribute, all copyright, patent, trademark, and
          attribution notices from the Source form of the Work,
          excluding those notices that do not pertain to any part of
          the Derivative Works; and

      (d) If the Work includes a "NOTICE" text file as part of its
          distribution, then any Derivative Works that You distribute must
          include a readable copy of the attribution notices contained
          within such NOTICE file, excluding those notices that do not
          pertain to any part of the Derivative Works, in at least one
          of the following places: within a NOTICE text file distributed
          as part of the Derivative Works; within the Source form or
          documentation, if provided along with the Derivative Works; or,
          within a display generated by the Derivative Works, if and
          wherever such third-party notices normally appear. The contents
          of the NOTICE file are for informational purposes only and
          do not modify the License. You may add Your own attribution
          notices within Derivative Works that You distribute, alongside
          or as an addendum to the NOTICE text from the Work, provided
          that such additional attribution notices cannot be construed
          as modifying the License.

      You may add Your own copyright statement to Your modifications and
      may provide additional or different license terms and conditions
      for use, reproduction, or distribution of Your modifications, or
      for any such Derivative Works as a whole, provided Your use,
      reproduction, and distribution of the Work otherwise complies with
      the conditions stated in this License.

   5. Submission of Contributions. Unless You explicitly state otherwise,
      any Contribution intentionally submitted for inclusion in the Work
      by You to the Licensor shall be under the terms and conditions of
      this License, without any additional terms or conditions.
      Notwithstanding the above, nothing herein shall supersede or modify
      the terms of any separate license agreement you may have executed
      with Licensor regarding such Contributions.

   6. Trademarks. This License does not grant permission to use the trade
      names, trademarks, service marks, or product names of the Licensor,
      except as required for reasonable and customary use in describing the
      origin of the Work and reproducing the content of the NOTICE file.

   7. Disclaimer of Warranty. Unless required by applicable law or
      agreed to in writing, Licensor provides the Work (and each
      Contributor provides its Contributions) on an "AS IS" BASIS,
      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
      implied, including, without limitation, any warranties or conditions
      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A
      PARTICULAR PURPOSE. You are solely responsible for determining the
      appropriateness of using or redistributing the Work and assume any
      risks associated with Your exercise of permissions under this License.

   8. Limitation of Liability. In no event and under no legal theory,
      whether in tort (including negligence), contract, or otherwise,
      unless required by applicable law (such as deliberate and grossly
      negligent acts) or agreed to in writing, shall any Contributor be
      liable to You for damages, including any direct, indirect, special,
      incidental, or consequential damages of any character arising as a
      result of this License or out of the use or inability to use the
      Work (including but not limited to damages for loss of goodwill,
      work stoppage, computer failure or malfunction, or any and all
      other commercial damages or losses), even if such Contributor
      has been advised of the possibility of such damages.

   9. Accepting Warranty or Additional Liability. While redistributing
      the Work or Derivative Works thereof, You may choose to offer,
      and charge a fee for, acceptance of support, warranty, indemnity,
      or other liability obligations and/or rights consistent with this
      License. However, in accepting such obligations, You may act only
      on Your own behalf and on Your sole responsibility, not on behalf
      of any other Contributor, and only if You agree to indemnify,
      defend, and hold each Contributor harmless for any liability
      incurred by, or claims asserted against, such Contributor by reason
      of your accepting any such warranty or additional liability.

   END OF TERMS AND CONDITIONS

   APPENDIX: How to apply the Apache License to your work.

      To apply the Apache License to your work, attach the following
      boilerplate notice, with the fields enclosed by brackets "[]"
      replaced with your own identifying information. (Don't include
      the brackets!)  The text should be enclosed in the appropriate
      comment syntax for the file format. We also recommend that a
      file or class name and description of purpose be included on the
      same "printed page" as the copyright notice for easier
      identification within third-party archives.

   Copyright [yyyy] [name of copyright owner]

   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at

       http://www.apache.org/licenses/LICENSE-2.0

   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License.'''

shutil.rmtree(BUNDLE_DIR, ignore_errors=True)
BUNDLE_DIR.mkdir(parents=True)
def write_file(relative_path, content):
    path = BUNDLE_DIR / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content.rstrip() + '\n', encoding='utf-8')

sampling_yaml = yaml.safe_dump(SAMPLING, sort_keys=False) if yaml else '''temperature: 0.2
top_p: 0.95
top_k: 40
max_output_tokens: 8192
thinking_config:
  thinking_budget: 4096
  include_thoughts: false
'''
write_file('configs/sampling.yaml', sampling_yaml)
write_file('prompts/system.md', SYSTEM_PROMPT)
write_file('NOTICE.txt', NOTICE)
write_file('LICENSE-APACHE-2.0.txt', LICENSE_TEXT)

analyzer_tools = ['run_command', 'read_file', 'search_similar_code', 'get_code_neighbors', 'get_code_subgraph']
coder_tools = ['run_command', 'read_file', 'edit_file', 'write_file', 'get_status',
               'submit_patch', 'search_similar_code', 'get_code_neighbors', 'get_code_subgraph']
analyzer_yaml = [
    'name: code_analyzer',
    f'model: {MODEL}',
    'description: Read-only code navigator returning candidate locations, root cause, and a fix plan.',
    'instruction: !include analyzer.md',
    'generate_content_config:',
    *['  ' + line for line in sampling_yaml.rstrip().splitlines()],
    'tools:',
    *[f'  - {tool}' for tool in analyzer_tools],
]
write_file('sub_agents/analyzer.md', ANALYZER_PROMPT)
write_file('sub_agents/code_analyzer.yaml', '\n'.join(analyzer_yaml))
agent_yaml = [
    'name: pathfinder_adapted',
    f'model: {MODEL}',
    'description: Software engineer that fixes one repository issue with a small, verified patch.',
    'instruction: !include prompts/system.md',
    'generate_content_config: !include configs/sampling.yaml',
    'tools:',
    *[f'  - {tool}' for tool in coder_tools],
    '  - agent_tool:',
    '      config_path: sub_agents/code_analyzer.yaml',
    '      skip_summarization: true',
]
write_file('agent.yaml', '\n'.join(agent_yaml))
evaluation_yaml = yaml.safe_dump({'evaluation': EVALUATION}, sort_keys=False) if yaml else '''evaluation:
  timeout_seconds: 180
  max_tool_calls: 40
  max_time_minutes: 4.5
  max_turns: 80
'''
write_file('eval_config.yaml', evaluation_yaml)
print('Bundle prepared:', BUNDLE_DIR)
print('Files:', sorted(str(p.relative_to(BUNDLE_DIR)) for p in BUNDLE_DIR.rglob('*') if p.is_file()))


Bundle prepared: /Users/flexonafft/MLKaggleTasks/competitions/public-comp/gemma-4-developer-agent/artifacts/analyzer_navigation_budgeted/gemma4_navigation_budget_bundle
Files: ['LICENSE-APACHE-2.0.txt', 'NOTICE.txt', 'agent.yaml', 'configs/sampling.yaml', 'eval_config.yaml', 'prompts/system.md', 'sub_agents/analyzer.md', 'sub_agents/code_analyzer.yaml']


In [5]:
'''Validate include resolution, model and tool permissions, and archive contents.'''

if yaml:
    class IncludeLoader(yaml.SafeLoader):
        pass
    IncludeLoader.add_constructor('!include', lambda loader, node: {'__include__': loader.construct_scalar(node)})

    def resolve(value, base, root, problems):
        if isinstance(value, dict) and set(value) == {'__include__'}:
            relative = value['__include__']
            target = (base / relative).resolve()
            if Path(relative).is_absolute() or '..' in Path(relative).parts or root not in target.parents or not target.is_file():
                problems.append(f'Invalid include: {relative}')
                return None
            if target.suffix in ('.yaml', '.yml'):
                return resolve(yaml.load(target.read_text(encoding='utf-8'), Loader=IncludeLoader), target.parent, root, problems)
            return target.read_text(encoding='utf-8')
        if isinstance(value, dict):
            return {key: resolve(item, base, root, problems) for key, item in value.items()}
        if isinstance(value, list):
            return [resolve(item, base, root, problems) for item in value]
        return value

    problems = []
    root = BUNDLE_DIR.resolve()
    agent = resolve(yaml.load((BUNDLE_DIR / 'agent.yaml').read_text(), Loader=IncludeLoader), BUNDLE_DIR, root, problems)
    analyzer = resolve(yaml.load((BUNDLE_DIR / 'sub_agents/code_analyzer.yaml').read_text(), Loader=IncludeLoader), BUNDLE_DIR / 'sub_agents', root, problems)
    assert not problems, problems
    assert agent['model'] == analyzer['model'] == MODEL
    assert 'submit_patch' in agent['tools'] and 'submit_patch' not in analyzer['tools']
    assert 'edit_file' not in analyzer['tools'] and 'write_file' not in analyzer['tools']
    assert agent['generate_content_config'] == analyzer['generate_content_config'] == SAMPLING
    assert set(t for t in agent['tools'] if isinstance(t, str)) == set(coder_tools)
    assert set(analyzer['tools']) == set(analyzer_tools)
    assert yaml.safe_load((BUNDLE_DIR / 'eval_config.yaml').read_text()) == {'evaluation': EVALUATION}
    assert set(EVALUATION) == {'timeout_seconds', 'max_tool_calls', 'max_time_minutes', 'max_turns'}
    assert 0 < EVALUATION['timeout_seconds'] <= EVALUATION['max_time_minutes'] * 60
    if importlib.util.find_spec('adk_submission'):
        from adk_submission.schema import SandboxedAgentConfig
        SandboxedAgentConfig.model_validate(agent)
        SandboxedAgentConfig.model_validate(analyzer)
        print('Official ADK schema check passed.')
    else:
        print('Official ADK schema check skipped: competition runtime package is unavailable locally.')
else:
    print('YAML checks skipped: PyYAML is unavailable.')

assert 'code_analyzer' in SYSTEM_PROMPT and 'You never modify files' in ANALYZER_PROMPT
assert '{' not in SYSTEM_PROMPT and '}' not in SYSTEM_PROMPT
print('Prompt/config checks passed.')


Official ADK schema check skipped: competition runtime package is unavailable locally.
Prompt/config checks passed.


In [6]:
'''Write a reproducible submission.zip and check the exact archive payload.'''

ZIP_PATH.unlink(missing_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(p for p in BUNDLE_DIR.rglob('*') if p.is_file()):
        entry = zipfile.ZipInfo(path.relative_to(BUNDLE_DIR).as_posix(), date_time=(1980, 1, 1, 0, 0, 0))
        entry.compress_type = zipfile.ZIP_DEFLATED
        entry.external_attr = 0o100644 << 16
        archive.writestr(entry, path.read_bytes())
with zipfile.ZipFile(ZIP_PATH) as archive:
    assert archive.testzip() is None
    names = set(archive.namelist())
    required = {
        'agent.yaml', 'configs/sampling.yaml', 'prompts/system.md',
        'sub_agents/analyzer.md', 'sub_agents/code_analyzer.yaml',
        'NOTICE.txt', 'LICENSE-APACHE-2.0.txt', 'eval_config.yaml',
    }
    assert required == names, sorted(required.symmetric_difference(names))
    assert all(not name.startswith('submission_bundle/') for name in names)

print('Created:', ZIP_PATH)
print('Bytes:', ZIP_PATH.stat().st_size)
print('SHA-256:', hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest())
print('Files:', sorted(names))
print('Package checks do not replace a Kaggle harness run.')


Created: /Users/flexonafft/MLKaggleTasks/competitions/public-comp/gemma-4-developer-agent/artifacts/analyzer_navigation_budgeted/submission.zip
Bytes: 8014
SHA-256: a69ca29bfa44a213867c0efe1a30d090bc731e8d029d5286729e5521e4d7faa6
Files: ['LICENSE-APACHE-2.0.txt', 'NOTICE.txt', 'agent.yaml', 'configs/sampling.yaml', 'eval_config.yaml', 'prompts/system.md', 'sub_agents/analyzer.md', 'sub_agents/code_analyzer.yaml']
Package checks do not replace a Kaggle harness run.
